# Smart Farm IoT - Dashboard de Monitorizacion

Dashboard interactivo para visualizar datos de sensores IoT de la granja inteligente.

**Sensores monitorizados:**
- Temperatura y Humedad (3 zonas, 9 sensores)
- Niveles de CO2 (por sensor)
- Humedad del Suelo (por sensor)

**Fuente de datos:** Delta Lake tables generadas por Spark Structured Streaming desde Kafka.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ruta base de datos (volumen montado desde ./tmp del host)
DATA_PATH = Path("/home/jovyan/data")

# Mapeo de sensores a zonas (igual que en el proyecto Scala)
SENSOR_ZONE_MAP = {
    "sensor1": "zone1", "sensor2": "zone1", "sensor3": "zone1",
    "sensor4": "zone2", "sensor5": "zone2", "sensor6": "zone2",
    "sensor7": "zone3", "sensor8": "zone3", "sensor9": "zone3",
}

ZONE_COLORS = {"zone1": "#2ecc71", "zone2": "#3498db", "zone3": "#e74c3c"}
ZONE_NAMES = {"zone1": "Zona 1 (Norte)", "zone2": "Zona 2 (Centro)", "zone3": "Zona 3 (Sur)"}

print("Imports cargados correctamente.")

Imports cargados correctamente.


## Carga de Datos

Intenta cargar datos desde las tablas Delta Lake generadas por el pipeline de Spark.  
Si no existen, genera datos de ejemplo realistas para demostrar las visualizaciones.

In [2]:
def load_delta_table(table_path: Path) -> pd.DataFrame | None:
    """Intenta cargar una tabla Delta Lake usando deltalake (delta-rs)."""
    if not table_path.exists():
        return None
    try:
        from deltalake import DeltaTable
        dt = DeltaTable(str(table_path))
        return dt.to_pandas()
    except Exception:
        # Fallback: leer archivos parquet directamente
        parquet_files = list(table_path.glob("**/*.parquet"))
        if parquet_files:
            dfs = [pd.read_parquet(f) for f in parquet_files]
            return pd.concat(dfs, ignore_index=True)
    return None


def generate_sample_data(hours: int = 6, interval_seconds: int = 10) -> dict:
    """Genera datos realistas de sensores IoT para demo."""
    np.random.seed(42)
    end_time = datetime.now()
    start_time = end_time - timedelta(hours=hours)
    timestamps = pd.date_range(start=start_time, end=end_time, freq=f"{interval_seconds}s")
    n_points = len(timestamps)

    # Perfiles base por zona (simulan microclimas diferentes)
    zone_profiles = {
        "zone1": {"temp_base": 22, "temp_amp": 5, "hum_base": 55, "co2_base": 400, "soil_base": 65},
        "zone2": {"temp_base": 25, "temp_amp": 4, "hum_base": 48, "co2_base": 420, "soil_base": 58},
        "zone3": {"temp_base": 20, "temp_amp": 6, "hum_base": 62, "co2_base": 380, "soil_base": 72},
    }

    temp_hum_rows = []
    co2_rows = []
    soil_rows = []

    for sensor_id, zone_id in SENSOR_ZONE_MAP.items():
        p = zone_profiles[zone_id]
        # Ciclo diurno para temperatura
        hours_array = np.array([(t.hour + t.minute / 60) for t in timestamps])
        temp_cycle = p["temp_amp"] * np.sin((hours_array - 6) * np.pi / 12)
        noise_t = np.random.normal(0, 0.5, n_points)
        noise_h = np.random.normal(0, 2, n_points)
        noise_co2 = np.random.normal(0, 10, n_points)
        noise_soil = np.random.normal(0, 1.5, n_points)

        # Drift lento del sensor (simulacion realista)
        drift = np.cumsum(np.random.normal(0, 0.01, n_points))

        temperatures = p["temp_base"] + temp_cycle + noise_t + drift
        humidities = p["hum_base"] - temp_cycle * 1.5 + noise_h  # humedad inversamente correlacionada
        co2_levels = p["co2_base"] + noise_co2 + 20 * np.sin(hours_array * np.pi / 12)
        soil_moistures = p["soil_base"] + noise_soil - np.linspace(0, 5, n_points)  # decae entre riegos

        # Simular evento de riego cada 2 horas
        for i in range(n_points):
            if timestamps[i].minute == 0 and timestamps[i].hour % 2 == 0 and timestamps[i].second == 0:
                soil_moistures[i:min(i+30, n_points)] += 15

        for i in range(n_points):
            ts = timestamps[i]
            temp_hum_rows.append({
                "sensorId": sensor_id,
                "temperature": round(float(temperatures[i]), 2),
                "humidity": round(float(np.clip(humidities[i], 20, 95)), 2),
                "timestamp": ts,
                "zoneId": zone_id,
            })
            co2_rows.append({
                "sensorId": sensor_id,
                "co2Level": round(float(np.clip(co2_levels[i], 300, 600)), 2),
                "timestamp": ts,
                "zoneId": zone_id,
            })
            soil_rows.append({
                "sensorId": sensor_id,
                "soilMoisture": round(float(np.clip(soil_moistures[i], 20, 95)), 2),
                "timestamp": ts,
            })

    return {
        "temp_hum": pd.DataFrame(temp_hum_rows),
        "co2": pd.DataFrame(co2_rows),
        "soil": pd.DataFrame(soil_rows),
    }


# Intentar cargar datos reales
df_temp_hum = load_delta_table(DATA_PATH / "temperature_humidity_zone_merge")
using_real_data = df_temp_hum is not None and len(df_temp_hum) > 0

if using_real_data:
    print(f"Datos REALES cargados desde Delta Lake: {len(df_temp_hum)} registros de temperatura/humedad")
    df_temp_hum["timestamp"] = pd.to_datetime(df_temp_hum["timestamp"])
    # CO2 y soil pueden no tener tablas Delta aun
    df_co2 = load_delta_table(DATA_PATH / "co2_zone_merge")
    df_soil = load_delta_table(DATA_PATH / "soil_moisture_merge")
    if df_co2 is None or df_soil is None:
        sample = generate_sample_data()
        if df_co2 is None:
            df_co2 = sample["co2"]
            print("CO2: usando datos de ejemplo (tabla Delta no encontrada)")
        if df_soil is None:
            df_soil = sample["soil"]
            print("Soil: usando datos de ejemplo (tabla Delta no encontrada)")
else:
    print("No se encontraron tablas Delta Lake en /home/jovyan/data/")
    print("Generando datos de ejemplo realistas para demo...")
    sample = generate_sample_data()
    df_temp_hum = sample["temp_hum"]
    df_co2 = sample["co2"]
    df_soil = sample["soil"]
    print(f"Generados: {len(df_temp_hum)} registros temp/hum, {len(df_co2)} CO2, {len(df_soil)} soil")

# Asegurar tipos
df_temp_hum["timestamp"] = pd.to_datetime(df_temp_hum["timestamp"])
df_co2["timestamp"] = pd.to_datetime(df_co2["timestamp"])
df_soil["timestamp"] = pd.to_datetime(df_soil["timestamp"])

No se encontraron tablas Delta Lake en /home/jovyan/data/
Generando datos de ejemplo realistas para demo...


Generados: 19449 registros temp/hum, 19449 CO2, 19449 soil


---
## Temperatura por Zona

Media de temperatura agrupada por zona con bandas de desviacion tipica.

In [3]:
# Resamplear a 1 minuto por zona para suavizar
df_temp_zone = (
    df_temp_hum
    .set_index("timestamp")
    .groupby("zoneId")
    .resample("1min")["temperature"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

fig_temp = go.Figure()

for zone_id in sorted(df_temp_zone["zoneId"].unique()):
    zone_data = df_temp_zone[df_temp_zone["zoneId"] == zone_id]
    color = ZONE_COLORS.get(zone_id, "#95a5a6")
    name = ZONE_NAMES.get(zone_id, zone_id)

    # Banda de desviacion tipica
    fig_temp.add_trace(go.Scatter(
        x=pd.concat([zone_data["timestamp"], zone_data["timestamp"][::-1]]),
        y=pd.concat([zone_data["mean"] + zone_data["std"], (zone_data["mean"] - zone_data["std"])[::-1]]),
        fill="toself",
        fillcolor=color.replace(")", ", 0.15)").replace("#", "rgba(") if color.startswith("rgba") else f"rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, 0.15)",
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
        hoverinfo="skip",
    ))
    # Linea media
    fig_temp.add_trace(go.Scatter(
        x=zone_data["timestamp"],
        y=zone_data["mean"],
        mode="lines",
        name=name,
        line=dict(color=color, width=2),
        hovertemplate="%{y:.1f} C<br>%{x}<extra>" + name + "</extra>",
    ))

fig_temp.update_layout(
    title="Temperatura Media por Zona",
    xaxis_title="Tiempo",
    yaxis_title="Temperatura (C)",
    template="plotly_white",
    height=450,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_temp.show()

---
## Humedad por Zona

In [4]:
df_hum_zone = (
    df_temp_hum
    .set_index("timestamp")
    .groupby("zoneId")
    .resample("1min")["humidity"]
    .agg(["mean", "std"])
    .reset_index()
)

fig_hum = go.Figure()

for zone_id in sorted(df_hum_zone["zoneId"].unique()):
    zone_data = df_hum_zone[df_hum_zone["zoneId"] == zone_id]
    color = ZONE_COLORS.get(zone_id, "#95a5a6")
    name = ZONE_NAMES.get(zone_id, zone_id)

    fig_hum.add_trace(go.Scatter(
        x=pd.concat([zone_data["timestamp"], zone_data["timestamp"][::-1]]),
        y=pd.concat([zone_data["mean"] + zone_data["std"], (zone_data["mean"] - zone_data["std"])[::-1]]),
        fill="toself",
        fillcolor=f"rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, 0.15)",
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
        hoverinfo="skip",
    ))
    fig_hum.add_trace(go.Scatter(
        x=zone_data["timestamp"],
        y=zone_data["mean"],
        mode="lines",
        name=name,
        line=dict(color=color, width=2),
        hovertemplate="%{y:.1f}%<br>%{x}<extra>" + name + "</extra>",
    ))

fig_hum.update_layout(
    title="Humedad Relativa Media por Zona",
    xaxis_title="Tiempo",
    yaxis_title="Humedad (%)",
    template="plotly_white",
    height=450,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_hum.show()

---
## Correlacion Temperatura vs Humedad

Scatter plot para identificar la relacion inversa entre temperatura y humedad ambiental.

In [5]:
# Muestra aleatoria para no saturar el grafico
sample_size = min(5000, len(df_temp_hum))
df_scatter = df_temp_hum.sample(n=sample_size, random_state=42)

fig_scatter = px.scatter(
    df_scatter,
    x="temperature",
    y="humidity",
    color="zoneId",
    color_discrete_map=ZONE_COLORS,
    opacity=0.5,
    labels={"temperature": "Temperatura (C)", "humidity": "Humedad (%)", "zoneId": "Zona"},
    title="Correlacion Temperatura vs Humedad",
    template="plotly_white",
    height=500,
    hover_data=["sensorId"],
)

# Linea de tendencia por zona
for zone_id in sorted(df_scatter["zoneId"].unique()):
    zd = df_scatter[df_scatter["zoneId"] == zone_id]
    z = np.polyfit(zd["temperature"], zd["humidity"], 1)
    p = np.poly1d(z)
    x_range = np.linspace(zd["temperature"].min(), zd["temperature"].max(), 50)
    fig_scatter.add_trace(go.Scatter(
        x=x_range, y=p(x_range),
        mode="lines",
        line=dict(color=ZONE_COLORS.get(zone_id), width=2, dash="dash"),
        showlegend=False,
        hoverinfo="skip",
    ))

fig_scatter.show()

---
## Niveles de CO2 por Sensor

Evolucion temporal del CO2 con umbrales de alerta.

In [6]:
# Resamplear CO2 a 1 minuto por sensor
df_co2_resampled = (
    df_co2
    .set_index("timestamp")
    .groupby("sensorId")
    .resample("1min")["co2Level"]
    .mean()
    .reset_index()
)

# Asignar zona
df_co2_resampled["zoneId"] = df_co2_resampled["sensorId"].map(SENSOR_ZONE_MAP)

fig_co2 = px.line(
    df_co2_resampled,
    x="timestamp",
    y="co2Level",
    color="sensorId",
    facet_row="zoneId",
    labels={"co2Level": "CO2 (ppm)", "timestamp": "Tiempo", "sensorId": "Sensor"},
    title="Niveles de CO2 por Sensor y Zona",
    template="plotly_white",
    height=700,
)

# Umbrales de CO2
for i in range(len(fig_co2.data)):
    fig_co2.data[i].line.width = 1.5

fig_co2.add_hline(y=450, line_dash="dash", line_color="orange",
                  annotation_text="Alerta (450 ppm)", annotation_position="top left")
fig_co2.add_hline(y=500, line_dash="dash", line_color="red",
                  annotation_text="Critico (500 ppm)", annotation_position="top left")

fig_co2.update_layout(hovermode="x unified")
fig_co2.show()

---
## Humedad del Suelo

Evolucion de la humedad del suelo con deteccion de eventos de riego.

In [7]:
df_soil_resampled = (
    df_soil
    .set_index("timestamp")
    .groupby("sensorId")
    .resample("1min")["soilMoisture"]
    .mean()
    .reset_index()
)

df_soil_resampled["zoneId"] = df_soil_resampled["sensorId"].map(SENSOR_ZONE_MAP)

fig_soil = go.Figure()

for sensor_id in sorted(df_soil_resampled["sensorId"].unique()):
    sd = df_soil_resampled[df_soil_resampled["sensorId"] == sensor_id]
    zone = SENSOR_ZONE_MAP.get(sensor_id, "unknown")
    fig_soil.add_trace(go.Scatter(
        x=sd["timestamp"],
        y=sd["soilMoisture"],
        mode="lines",
        name=f"{sensor_id} ({zone})",
        line=dict(width=1.5),
        hovertemplate="%{y:.1f}%<br>%{x}<extra>" + sensor_id + "</extra>",
    ))

# Zona optima de humedad del suelo
fig_soil.add_hrect(y0=50, y1=75, fillcolor="green", opacity=0.08,
                   annotation_text="Rango optimo", annotation_position="top left")
fig_soil.add_hline(y=35, line_dash="dot", line_color="red",
                   annotation_text="Riego necesario", annotation_position="bottom left")

fig_soil.update_layout(
    title="Humedad del Suelo por Sensor",
    xaxis_title="Tiempo",
    yaxis_title="Humedad del Suelo (%)",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_soil.show()

---
## Distribucion de Lecturas por Zona

Box plots para comparar la distribucion de cada metrica entre zonas.

In [8]:
fig_dist = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Temperatura (C)", "Humedad (%)", "CO2 (ppm)"),
    horizontal_spacing=0.08,
)

for zone_id in sorted(df_temp_hum["zoneId"].unique()):
    color = ZONE_COLORS.get(zone_id, "#95a5a6")
    name = ZONE_NAMES.get(zone_id, zone_id)
    zd_th = df_temp_hum[df_temp_hum["zoneId"] == zone_id]

    fig_dist.add_trace(go.Box(
        y=zd_th["temperature"], name=name, marker_color=color,
        legendgroup=zone_id, showlegend=True,
    ), row=1, col=1)

    fig_dist.add_trace(go.Box(
        y=zd_th["humidity"], name=name, marker_color=color,
        legendgroup=zone_id, showlegend=False,
    ), row=1, col=2)

    zd_co2 = df_co2[df_co2.get("zoneId", pd.Series(dtype=str)).isin([zone_id])]
    if len(zd_co2) == 0 and "sensorId" in df_co2.columns:
        sensors_in_zone = [s for s, z in SENSOR_ZONE_MAP.items() if z == zone_id]
        zd_co2 = df_co2[df_co2["sensorId"].isin(sensors_in_zone)]

    fig_dist.add_trace(go.Box(
        y=zd_co2["co2Level"], name=name, marker_color=color,
        legendgroup=zone_id, showlegend=False,
    ), row=1, col=3)

fig_dist.update_layout(
    title="Distribucion de Lecturas por Zona",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
)
fig_dist.show()

---
## Dashboard Combinado

Vista consolidada con indicadores clave y todas las metricas.

In [9]:
# Calcular metricas actuales (ultimo minuto)
last_minute = df_temp_hum["timestamp"].max() - timedelta(minutes=1)
recent = df_temp_hum[df_temp_hum["timestamp"] >= last_minute]

avg_temp = recent["temperature"].mean()
avg_hum = recent["humidity"].mean()
avg_co2 = df_co2[df_co2["timestamp"] >= last_minute]["co2Level"].mean() if len(df_co2) > 0 else 0
avg_soil = df_soil[df_soil["timestamp"] >= last_minute]["soilMoisture"].mean() if len(df_soil) > 0 else 0

fig_dashboard = make_subplots(
    rows=3, cols=4,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"type": "scatter", "colspan": 2}, None, {"type": "scatter", "colspan": 2}, None],
        [{"type": "scatter", "colspan": 2}, None, {"type": "scatter", "colspan": 2}, None],
    ],
    subplot_titles=("", "", "", "", "Temperatura por Zona", "CO2 por Zona", "Humedad por Zona", "Humedad del Suelo"),
    row_heights=[0.2, 0.4, 0.4],
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
)

# Indicadores KPI
fig_dashboard.add_trace(go.Indicator(
    mode="gauge+number",
    value=avg_temp,
    title={"text": "Temp (C)"},
    gauge={"axis": {"range": [0, 45]},
           "bar": {"color": "#e74c3c"},
           "steps": [{"range": [0, 15], "color": "#d4efdf"},
                     {"range": [15, 30], "color": "#abebc6"},
                     {"range": [30, 45], "color": "#f5b7b1"}]},
), row=1, col=1)

fig_dashboard.add_trace(go.Indicator(
    mode="gauge+number",
    value=avg_hum,
    title={"text": "Humedad (%)"},
    gauge={"axis": {"range": [0, 100]},
           "bar": {"color": "#3498db"},
           "steps": [{"range": [0, 30], "color": "#f9e79f"},
                     {"range": [30, 70], "color": "#aed6f1"},
                     {"range": [70, 100], "color": "#a9cce3"}]},
), row=1, col=2)

fig_dashboard.add_trace(go.Indicator(
    mode="gauge+number",
    value=avg_co2,
    title={"text": "CO2 (ppm)"},
    gauge={"axis": {"range": [300, 600]},
           "bar": {"color": "#f39c12"},
           "steps": [{"range": [300, 400], "color": "#d5f5e3"},
                     {"range": [400, 500], "color": "#fdebd0"},
                     {"range": [500, 600], "color": "#f5b7b1"}]},
), row=1, col=3)

fig_dashboard.add_trace(go.Indicator(
    mode="gauge+number",
    value=avg_soil,
    title={"text": "Suelo (%)"},
    gauge={"axis": {"range": [0, 100]},
           "bar": {"color": "#27ae60"},
           "steps": [{"range": [0, 35], "color": "#f5b7b1"},
                     {"range": [35, 75], "color": "#abebc6"},
                     {"range": [75, 100], "color": "#aed6f1"}]},
), row=1, col=4)

# Series temporales resampleadas a 5 min para el dashboard
for zone_id in sorted(df_temp_hum["zoneId"].unique()):
    color = ZONE_COLORS.get(zone_id, "#95a5a6")
    name = ZONE_NAMES.get(zone_id, zone_id)
    zd = df_temp_hum[df_temp_hum["zoneId"] == zone_id].set_index("timestamp").resample("5min")

    temp_5m = zd["temperature"].mean().reset_index()
    hum_5m = zd["humidity"].mean().reset_index()

    fig_dashboard.add_trace(go.Scatter(
        x=temp_5m["timestamp"], y=temp_5m["temperature"],
        mode="lines", name=name, line=dict(color=color, width=1.5),
        legendgroup=zone_id, showlegend=True,
    ), row=2, col=1)

    fig_dashboard.add_trace(go.Scatter(
        x=hum_5m["timestamp"], y=hum_5m["humidity"],
        mode="lines", name=name, line=dict(color=color, width=1.5),
        legendgroup=zone_id, showlegend=False,
    ), row=3, col=1)

    # CO2 por zona
    sensors = [s for s, z in SENSOR_ZONE_MAP.items() if z == zone_id]
    co2_zone = df_co2[df_co2["sensorId"].isin(sensors)].set_index("timestamp").resample("5min")["co2Level"].mean().reset_index()
    fig_dashboard.add_trace(go.Scatter(
        x=co2_zone["timestamp"], y=co2_zone["co2Level"],
        mode="lines", name=name, line=dict(color=color, width=1.5),
        legendgroup=zone_id, showlegend=False,
    ), row=2, col=3)

    # Soil por zona
    soil_zone = df_soil[df_soil["sensorId"].isin(sensors)].set_index("timestamp").resample("5min")["soilMoisture"].mean().reset_index()
    fig_dashboard.add_trace(go.Scatter(
        x=soil_zone["timestamp"], y=soil_zone["soilMoisture"],
        mode="lines", name=name, line=dict(color=color, width=1.5),
        legendgroup=zone_id, showlegend=False,
    ), row=3, col=3)

fig_dashboard.update_layout(
    title="Smart Farm IoT - Dashboard de Monitorizacion",
    template="plotly_white",
    height=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)
fig_dashboard.show()

---
## Tabla Resumen Estadistico

In [10]:
# Estadisticas por zona
stats = []
for zone_id in sorted(ZONE_NAMES.keys()):
    sensors = [s for s, z in SENSOR_ZONE_MAP.items() if z == zone_id]
    th = df_temp_hum[df_temp_hum["zoneId"] == zone_id]
    co2 = df_co2[df_co2["sensorId"].isin(sensors)]
    soil = df_soil[df_soil["sensorId"].isin(sensors)]

    stats.append({
        "Zona": ZONE_NAMES[zone_id],
        "Temp Media (C)": f"{th['temperature'].mean():.1f}",
        "Temp Min (C)": f"{th['temperature'].min():.1f}",
        "Temp Max (C)": f"{th['temperature'].max():.1f}",
        "Humedad Media (%)": f"{th['humidity'].mean():.1f}",
        "CO2 Medio (ppm)": f"{co2['co2Level'].mean():.1f}",
        "Suelo Medio (%)": f"{soil['soilMoisture'].mean():.1f}",
        "N Lecturas": f"{len(th):,}",
    })

df_stats = pd.DataFrame(stats)

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_stats.columns),
        fill_color="#2c3e50",
        font=dict(color="white", size=12),
        align="center",
    ),
    cells=dict(
        values=[df_stats[col] for col in df_stats.columns],
        fill_color=[["#d5f5e3", "#aed6f1", "#f5b7b1"]] * len(df_stats.columns),
        align="center",
        font=dict(size=11),
    ),
)])

fig_table.update_layout(
    title="Resumen Estadistico por Zona",
    height=250,
)
fig_table.show()

---
## Heatmap: Actividad de Sensores por Hora

Mapa de calor mostrando la temperatura media por sensor y hora del dia.

In [11]:
df_temp_hum["hour"] = df_temp_hum["timestamp"].dt.hour

heatmap_data = (
    df_temp_hum
    .groupby(["sensorId", "hour"])["temperature"]
    .mean()
    .reset_index()
    .pivot(index="sensorId", columns="hour", values="temperature")
    .fillna(0)
)

fig_heat = px.imshow(
    heatmap_data,
    labels=dict(x="Hora del Dia", y="Sensor", color="Temp (C)"),
    title="Temperatura Media por Sensor y Hora",
    color_continuous_scale="RdYlBu_r",
    aspect="auto",
    template="plotly_white",
    height=400,
)
fig_heat.show()

# Limpiar columna temporal
df_temp_hum.drop(columns=["hour"], inplace=True)

---
## Consumo en Tiempo Real desde Kafka (Opcional)

Esta celda se conecta directamente a Kafka para consumir datos en tiempo real.  
**Requiere** que el cluster de Kafka este levantado y el generador de datos en ejecucion.

Descomenta y ejecuta si quieres ver datos en streaming.

In [12]:
# from kafka import KafkaConsumer
# from IPython.display import clear_output
# import time
#
# KAFKA_BROKERS = ["kafka1:29092", "kafka2:29092", "kafka3:29092"]
#
# consumer = KafkaConsumer(
#     "temperature_humidity",
#     bootstrap_servers=KAFKA_BROKERS,
#     auto_offset_reset="latest",
#     group_id="jupyter-dashboard",
#     value_deserializer=lambda m: m.decode("utf-8"),
# )
#
# buffer = []
# fig_rt = go.FigureWidget()
# fig_rt.update_layout(title="Temperatura en Tiempo Real", template="plotly_white",
#                      xaxis_title="Muestra", yaxis_title="Temperatura (C)")
# display(fig_rt)
#
# try:
#     for message in consumer:
#         parts = message.value.split(",")
#         if len(parts) == 4:
#             sensor_id, temp, hum, ts = parts[0], float(parts[1]), float(parts[2]), parts[3]
#             buffer.append({"sensor": sensor_id, "temp": temp, "hum": hum, "ts": ts})
#             if len(buffer) > 500:
#                 buffer = buffer[-500:]
#             if len(buffer) % 10 == 0:  # Actualizar cada 10 mensajes
#                 df_rt = pd.DataFrame(buffer)
#                 with fig_rt.batch_update():
#                     fig_rt.data = []
#                     for sid in df_rt["sensor"].unique():
#                         sd = df_rt[df_rt["sensor"] == sid]
#                         fig_rt.add_scatter(x=list(range(len(sd))), y=sd["temp"].tolist(),
#                                           mode="lines", name=sid)
# except KeyboardInterrupt:
#     consumer.close()
#     print("Consumer detenido.")